# Comparative CSV: Human vs LLM-generated texts

This notebook produces comparative CSV files that combine **human-written** and **LLM-generated** texts:

1. **Abstracts (formal)** — from `sv_abstracts_generated.csv` (Abstract vs Generated_Abstract)
2. **Comments (informal)** — human Reddit comments + generated comments (same logic as `inspect_reddit_comments.py`)

In [ ]:
import os
import pandas as pd

# Paths: run from project root or from src/
cwd = os.getcwd()
SRC_DIR = cwd if os.path.isdir(os.path.join(cwd, "data_collection")) else os.path.join(cwd, "src")
DATA_DIR = os.path.join(SRC_DIR, "data_collection")
FORMAL_DIR = os.path.join(DATA_DIR, "formal")
INFORMAL_DIR = os.path.join(DATA_DIR, "informal")

# Input files
ABSTRACTS_CSV = os.path.join(FORMAL_DIR, "sv_abstracts_generated.csv")
COMMENTS_HUMAN_CSV = os.path.join(INFORMAL_DIR, "reddit_comments.csv")
COMMENTS_GENERATED_CSV = os.path.join(DATA_DIR, "reddit_comments_openai_100.csv")

# Output files (comparative CSVs)
OUT_ABSTRACTS_COMPARATIVE = os.path.join(FORMAL_DIR, "comparative_abstracts.csv")
OUT_COMMENTS_COMPARATIVE = os.path.join(DATA_DIR, "comparative_comments.csv")

print("Data dir:", DATA_DIR)
print("Abstracts input:", ABSTRACTS_CSV)
print("Comments human:", COMMENTS_HUMAN_CSV)
print("Comments generated:", COMMENTS_GENERATED_CSV)

: 

## 1. Abstracts (formal): human vs generated

Load `sv_abstracts_generated.csv` (already has both columns). We produce:
- **Wide format** (unchanged): one row per thesis with `Abstract` and `Generated_Abstract`.
- **Long format** (optional): one row per text with `source` = human | generated, for consistent analysis with comments.

In [ ]:
# Load formal abstracts (human + generated in same file)
df_abs = pd.read_csv(ABSTRACTS_CSV, encoding="utf-8")
print(f"Loaded {len(df_abs)} rows. Columns: {list(df_abs.columns)}")

# Keep key columns for comparative CSV (human vs LLM side by side)
cols_abstracts = [c for c in ["Year", "Level", "Title", "Topic Category", "Abstract", "Generated_Abstract"] if c in df_abs.columns]
df_abstracts_comparative = df_abs[cols_abstracts].copy()
df_abstracts_comparative.to_csv(OUT_ABSTRACTS_COMPARATIVE, index=False, encoding="utf-8")
print(f"Saved comparative abstracts (wide) to {OUT_ABSTRACTS_COMPARATIVE}")

In [ ]:
# Optional: long-format abstracts (one row per text, with source = human | generated)
rows_long = []
for idx, row in df_abs.iterrows():
    if pd.notna(row.get("Abstract")) and str(row["Abstract"]).strip():
        rows_long.append({
            "index": idx,
            "Year": row.get("Year"),
            "Title": row.get("Title"),
            "text": row["Abstract"],
            "source": "human",
        })
    if pd.notna(row.get("Generated_Abstract")) and str(row["Generated_Abstract"]).strip():
        rows_long.append({
            "index": idx,
            "Year": row.get("Year"),
            "Title": row.get("Title"),
            "text": row["Generated_Abstract"],
            "source": "generated",
        })
df_abstracts_long = pd.DataFrame(rows_long)
out_abstracts_long = os.path.join(FORMAL_DIR, "comparative_abstracts_long.csv")
df_abstracts_long.to_csv(out_abstracts_long, index=False, encoding="utf-8")
print(f"Saved long-format abstracts to {out_abstracts_long} ({len(df_abstracts_long)} rows)")
df_abstracts_long.head(6)

## 2. Comments (informal): human vs generated

Same logic as `inspect_reddit_comments.get_combined_df()`: load human and generated comment CSVs, add a `source` column, concatenate, and save one comparative CSV.

In [ ]:
# Load human and generated comments
human = pd.read_csv(COMMENTS_HUMAN_CSV, encoding="utf-8")
gen = pd.read_csv(COMMENTS_GENERATED_CSV, encoding="utf-8")
human["source"] = "human"
gen["source"] = "generated"

# Same columns in both (link_id, question, comment, source)
cols = [c for c in human.columns if c in gen.columns]
df_comments_comparative = pd.concat([human[cols], gen[cols]], ignore_index=True)
df_comments_comparative.to_csv(OUT_COMMENTS_COMPARATIVE, index=False, encoding="utf-8")
print(f"Human comments: {len(human)}, Generated: {len(gen)}")
print(f"Saved comparative comments to {OUT_COMMENTS_COMPARATIVE} ({len(df_comments_comparative)} rows)")
df_comments_comparative.head(10)

## Summary

- **Abstracts (wide):** `data_collection/formal/comparative_abstracts.csv` — Year, Level, Title, Topic Category, Abstract, Generated_Abstract  
- **Abstracts (long):** `data_collection/formal/comparative_abstracts_long.csv` — index, Year, Title, text, source  
- **Comments:** `data_collection/comparative_comments.csv` — link_id, question, comment, source